# Credit Risk Assessment - Complete Pipeline
## XGBoost + SHAP Interpretability

This notebook trains the complete model and generates all necessary artifacts.

In [ ]:
# Imports
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader import DataLoader
from preprocessor import CreditRiskPreprocessor, split_data
from model_trainer import CreditRiskModelTrainer
from explainer import CreditRiskExplainer
from risk_engine import CreditRiskEngine

print("✓ All modules imported successfully")

## 1. Load Data

In [ ]:
# Load data
loader = DataLoader()
df = loader.load_data()

print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nTarget distribution:")
print(df['default'].value_counts(normalize=True))

df.head()

## 2. Data Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = CreditRiskPreprocessor()

# Fit and transform
X, y = preprocessor.fit_transform(df, target_col='default')

print(f"\nPreprocessed shape: {X.shape}")
print(f"Features: {X.columns.tolist()[:10]}...")

# Split data
data_splits = split_data(X, y, test_size=0.2, val_size=0.1, random_state=42)

print(f"\nTrain: {len(data_splits['X_train'])}")
print(f"Val: {len(data_splits['X_val'])}")
print(f"Test: {len(data_splits['X_test'])}")

## 3. Train Models

In [ ]:
# Initialize trainer
trainer = CreditRiskModelTrainer()

# Train XGBoost (primary model)
print("\n" + "="*60)
print("Training XGBoost (Primary Model)")
print("="*60)

xgb_param_grid = {
    'max_depth': [3, 4, 5],
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
}

xgb_model = trainer.train_xgboost(
    data_splits['X_train'], data_splits['y_train'],
    data_splits['X_val'], data_splits['y_val'],
    param_grid=xgb_param_grid,
    cv=3
)

In [ ]:
# Train LightGBM (comparison)
print("\n" + "="*60)
print("Training LightGBM (Comparison)")
print("="*60)

lgb_param_grid = {
    'max_depth': [3, 4, 5],
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
}

lgb_model = trainer.train_lightgbm(
    data_splits['X_train'], data_splits['y_train'],
    data_splits['X_val'], data_splits['y_val'],
    param_grid=lgb_param_grid,
    cv=3
)

In [ ]:
# Train CatBoost (comparison)
print("\n" + "="*60)
print("Training CatBoost (Comparison)")
print("="*60)

cat_param_grid = {
    'depth': [3, 4, 5],
    'iterations': [100, 200],
    'learning_rate': [0.05, 0.1],
}

cat_model = trainer.train_catboost(
    data_splits['X_train'], data_splits['y_train'],
    data_splits['X_val'], data_splits['y_val'],
    param_grid=cat_param_grid,
    cv=3
)

## 4. Model Comparison

In [ ]:
# Compare models
comparison = trainer.compare_models()
display(comparison)

# Get best model
best_model_name, best_model = trainer.get_best_model()

# Plot feature importance
trainer.plot_feature_importance('xgboost', X.columns.tolist(), top_n=20)

## 5. SHAP Explainability

In [ ]:
# Initialize explainer with XGBoost model
explainer = CreditRiskExplainer(xgb_model, X.columns.tolist())

# Initialize SHAP
explainer.initialize_shap(data_splits['X_train'], max_samples=1000)

# Compute SHAP values for test set
shap_values = explainer.compute_shap_values(data_splits['X_test'])

print("✓ SHAP values computed")

In [ ]:
# SHAP summary plot
explainer.plot_shap_summary(data_splits['X_test'], max_display=20)

In [ ]:
# SHAP bar plot
explainer.plot_shap_bar(data_splits['X_test'], max_display=20)

In [ ]:
# Individual prediction explanation
instance_idx = 0
print(f"\nExplaining instance {instance_idx}:")
print(f"Actual: {data_splits['y_test'].iloc[instance_idx]}")
print(f"Predicted: {xgb_model.predict(data_splits['X_test'].iloc[[instance_idx]])[0]}")
print(f"Probability: {xgb_model.predict_proba(data_splits['X_test'].iloc[[instance_idx]])[0][1]:.4f}")

# Top features
top_features = explainer.get_top_features_for_instance(
    instance_idx, data_splits['X_test'], top_n=10
)
display(top_features)

## 6. Risk Engine

In [ ]:
# Initialize risk engine
risk_engine = CreditRiskEngine()

# Predict on test set
test_probabilities = xgb_model.predict_proba(data_splits['X_test'])[:, 1]

# Assess first few applicants
sample_assessments = risk_engine.batch_assess(test_probabilities[:100])

print("\nSample Risk Assessments:")
display(sample_assessments.head(10))

# Portfolio metrics
portfolio_metrics = risk_engine.calculate_portfolio_metrics(sample_assessments)

print("\nPortfolio Metrics:")
import json
print(json.dumps(portfolio_metrics, indent=2))

## 7. Save Artifacts

In [ ]:
# Save preprocessor
preprocessor.save('../models/preprocessor.pkl')

# Save SHAP background data
import joblib
background_sample = data_splits['X_train'].sample(min(1000, len(data_splits['X_train'])), random_state=42)
joblib.dump(background_sample, '../models/shap_background.pkl')

# Save explainer
explainer.save('../models/explainer.pkl')

print("\n✓ All artifacts saved to models/ directory")
print("\n🎉 Model training complete!")
print("\nNext steps:")
print("1. Start API: uvicorn api.main:app --reload")
print("2. Open frontend/index.html in browser")